# Sample for IAA

* 10% of the texts
* Exclude anything that might lead to exclusion (WPM, chatgpt, no consent, etc.)

In [1]:
import pandas as pd
import json
import os
import random

In [2]:
# Set random seed
# Every time you run the code, you will get the same random sample
# Note: this means running the code on the same computer, as there are other parts of the
# code (sets), that give different results on different machines

# The seed number doesn't matter, can be anything (this is just about fixing it). 
random.seed(10)

In [3]:
def create_text_dict():
    
    d = {
      "id": 5,
      "data": {
        "text": ""
      },
      "annotations": [],
      "predictions": []
            }
    return d

In [4]:
# load data from excel overview and get text identifiers of included texts,
# then sample 10%
path = '../results/kim-full-data-update1.xlsx'

df = pd.read_excel(path)

text_ids = set()
text_ids_total = set()

for i, row in df.iterrows():
    text_id = row['Article_id']
    cond = row['condition']
    art_id = f'{text_id} {cond}'
    text_ids_total.add(art_id)
    if not row['exclude'] != '-':
        text_ids.add(art_id)
        
print('Total number of texts in the dataset:')    
print(len(text_ids_total))
print('Number of certainly included texts:')
print(len(text_ids))


Total number of texts in the dataset:
1308
Number of certainly included texts:
1124


In [12]:
#sample

# Make LS input for IAA check

## 1. Span annotation (tc-reference)

In [11]:
# get random sample of 10 percent
n_texts = int(round(len(text_ids) * 0.1, 0))
print(n_texts)

sample = random.sample(list(text_ids), n_texts)
print(len(sample))

112
112


In [7]:
# Load original data


# Place original data (Qualtrics output) in '../data' and edit the name of the file below (if necessary)
path = '../data/final_dataset.xlsx'
df = pd.read_excel(path)
data = df.to_dict('records')
# print(len(data[4].keys()))
# print(data[4].keys())

In [8]:
# Code from input creation

json_list = []
data_dir = '../data'
  
c = 0
for d in data[1:]:
    part_id = d['ParticipantID']
    # att_check = d['Reject: attentioncheck wrong']
    # consent = d['Consent form']
    

    # if d['INCLUDE'] == 'TRUE':
    #     ls_ids = []

    for k, v in d.items():
        k_words = k.split(' ')
        if (k_words[-1] == 'txt' or k_words[-1] == 'txt.1') and not k.startswith('wc_'):
            if str(v) != 'nan' and v != 0:
                text_id = f'{part_id} {k}'
                text_id_part = f'output-{text_id}'

                if text_id_part in sample:
                    # print('found text')
                    text = f'{part_id} {k}\nText: {v}'

                    text_d = create_text_dict()

                    # text_id = d['id']
                    text_d['id'] =  text_id
                    # text = f"Text id: {text_id}\nText: {d['text']}"
                    text_d['data']['text'] = text
                    json_list.append(text_d)
                    c += 1
 
                
                

print(len(json_list))
    
json_str = json.dumps(json_list)


# JSON file for ls will be written to '../data/tasksIAA.json'
with open(f'{data_dir}/tasksIAA.json', 'w') as outfile:
    outfile.write(json_str)

112


## 2. TC-reference check-boxes

Input: texts annotated with tc-reference spans

In [13]:
# sample 10% again
n_texts = int(round(len(text_ids) * 0.1, 0))
print(n_texts)

sample = random.sample(list(text_ids), n_texts)
print(len(sample))

112
112


In [15]:
# get texts from the existing annotations
name = 'kim-full-data-update1'
output = 'data'
path = f'../{output}/{name}.json'
with open(path, encoding = 'utf-8') as infile:
    annotation_data = json.load(infile)

In [64]:
# print(sample)

In [38]:
sample_texts = []

for text_dict in annotation_data:
    text_data = text_dict['data']['text']

    text_id = text_data.split('\n')
    text_id = f'output-{text_id[0]}'

    if text_id in sample:
        sample_texts.append(text_dict)


In [39]:
print(len(sample_texts))

112


In [62]:
# remove all annotations except TC-reference spans
for text_dict in sample_texts:

    annotations_new = []
    annotations = text_dict['annotations'][0]['result']
    for ann in annotations:
        if 'value' in ann:
            if 'labels' in ann['value']:
                labels = ann['value']['labels']
                if 'TC-reference' in labels:
                    annotations_new.append(ann)
    text_dict['annotations'][0]['result'] = annotations_new


In [63]:
# Make LS file

json_str = json.dumps(sample_texts)

# JSON file for ls will be written to '../data/tasksIAA.json'
with open(f'{data_dir}/tasksIAA-TC-ref.json', 'w') as outfile:
    outfile.write(json_str)